<a href="https://colab.research.google.com/github/Omkar-Shetkar/deep-learning-lab/blob/main/small_language_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import requests

# Download the Tiny Shakespeare dataset
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

print(f"Length of dataset: {len(text)} characters")
print("First 100 characters:\n", text[:100])

Length of dataset: 1115394 characters
First 100 characters:
 First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You


In [5]:
# All unique characters that occur in this text
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"Unique characters: {''.join(chars)}")
print(f"Vocabulary size: {vocab_size}")

Unique characters: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Vocabulary size: 65


In [6]:
# Mapping characters to integers
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }

# Encoder: take a string, output a list of integers
encode = lambda s: [stoi[c] for c in s]

# Decoder: take a list of integers, output a string
decode = lambda l: ''.join([itos[i] for i in l])

# Test it!
test_str = "hello world"
encoded = encode(test_str)
print(f"Encoded: {encoded}")
print(f"Decoded: {decode(encoded)}")

Encoded: [46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
Decoded: hello world


In [7]:
import torch

# Encode the entire text and wrap it in a PyTorch Tensor
data = torch.tensor(encode(text), dtype=torch.long)

# Split into 90% train, 10% validation
n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print(f"Train size: {len(train_data)} tokens")
print(f"Val size: {len(val_data)} tokens")

Train size: 1003854 tokens
Val size: 111540 tokens


In [8]:
torch.manual_seed(1337) # For reproducibility
batch_size = 4 # How many independent sequences will we process in parallel?
block_size = 8 # What is the maximum context length for predictions?

def get_batch(split):
    # Generate a small batch of data of inputs x and targets y
    data_set = train_data if split == 'train' else val_data

    # Pick random starting points in the text
    ix = torch.randint(len(data_set) - block_size, (batch_size,))

    # Stack the chunks into a matrix (Batch Size x Block Size)
    x = torch.stack([data_set[i:i+block_size] for i in ix])

    # The targets (y) are just the same sequence shifted by 1
    y = torch.stack([data_set[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('Inputs (xb):')
print(xb.shape)
print(xb)
print('Targets (yb):')
print(yb.shape)
print(yb)

Inputs (xb):
torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
Targets (yb):
torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [9]:
import torch
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each token directly reads off the logits for the next token from a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (Batch, Block) tensors of integers
        logits = self.token_embedding_table(idx) # (Batch, Block, Vocab_size)

        if targets is None:
            loss = None
        else:
            # Reshape the data for PyTorch's cross_entropy function
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            # Get the predictions
            logits, loss = self(idx)
            # Focus only on the last time step
            logits = logits[:, -1, :] # becomes (B, C)
            # Apply softmax to get probabilities
            probs = F.softmax(logits, dim=-1) # (B, C)
            # Sample from the distribution
            idx_next = torch.multinomial(probs, num_samples=1) # (B, 1)
            # Append sampled index to the running sequence
            idx = torch.cat((idx, idx_next), dim=1) # (B, T+1)
        return idx

In [10]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [11]:
model = BigramLanguageModel(vocab_size)
m = model.to(device)

# Create a "starting" token (0 is usually a newline or space)
context = torch.zeros((1, 1), dtype=torch.long, device=device)

# Generate 100 characters
print(decode(m.generate(context, max_new_tokens=100)[0].tolist()))


yq$;tfBfROkNdcuwdZZTkOMl;,ertK
w:!PLCkMBbeA$3:XaSGJO-3p&M-c?KL3auhpFYVXJFhNNNuhq$OMxv.tbVFYdXlrFZaAe


In [12]:
# Create a PyTorch optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

batch_size = 32
for steps in range(3000): # Increase this for better results
    # Sample a batch of data
    xb, yb = get_batch('train')

    # Move data to the correct device
    xb = xb.to(device)
    yb = yb.to(device)

    # Evaluate the loss
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if steps % 500 == 0:
        print(f"step {steps}: loss {loss.item():.4f}")

print(f"Final loss: {loss.item():.4f}")

step 0: loss 4.6816
step 500: loss 4.1312
step 1000: loss 3.6277
step 1500: loss 3.3737
step 2000: loss 3.1561
step 2500: loss 3.0014
Final loss: 2.8594


In [13]:
head_size = 16 # The dimensionality of the "search space"

class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, input_embed_dim):
        super().__init__()
        # These are the "projections" (linear layers) for our search
        self.key = nn.Linear(input_embed_dim, head_size, bias=False)
        self.query = nn.Linear(input_embed_dim, head_size, bias=False)
        self.value = nn.Linear(input_embed_dim, head_size, bias=False)

        # 'tril' is a triangular mask so we don't "cheat" by looking at the future
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # Compute "affinity" (how much do these characters like each other?)
        # We multiply the queries by the keys
        wei = q @ k.transpose(-2, -1) * (head_size**-0.5) # (B, T, T)

        # Mask out the future (the model can't see characters that haven't happened yet)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1) # (B, T, T)

        # Perform the weighted aggregation of the values
        v = self.value(x) # (B, T, head_size)
        out = wei @ v # (B, T, head_size)
        return out

In [14]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # Each token looks up a vector of size 'n_embd'
        n_embd = 32 # The "meaning" dimension
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)

        # This is our new "Eyes" - the self-attention head
        # We tell it: "Look at vectors of size n_embd, and output vectors of size head_size"
        self.sa_head = Head(n_embd)

        # Final layer to convert vectors back to character scores
        # This layer now expects input of 'head_size' (the output of sa_head)
        self.lm_head = nn.Linear(head_size, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Step 1: Turn integers into vectors (B, T, C)
        tok_emb = self.token_embedding_table(idx)

        # Step 2: Let the characters "talk" to each other
        x = self.sa_head(tok_emb)

        # Step 3: Get the final scores for the next character
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # Crop the context so we don't exceed the block_size
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [15]:
# 1. Initialize the upgraded model
# Make sure your 'Head' class and new 'BigramLanguageModel' class are defined above this!
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BigramLanguageModel(vocab_size)
m = model.to(device)

# 2. Set up the optimizer for the new parameters
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

# 3. The Training Loop
max_iters = 5000
eval_interval = 500

for iter in range(max_iters):

    # Every now and then evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        print(f"step {iter}: computing loss...")

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Move data to the correct device
    xb = xb.to(device)
    yb = yb.to(device)

    # Evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"Final Loss: {loss.item():.4f}")

# 4. Generate something!
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("--- GENERATED TEXT ---")
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))

step 0: computing loss...
step 500: computing loss...
step 1000: computing loss...
step 1500: computing loss...
step 2000: computing loss...
step 2500: computing loss...
step 3000: computing loss...
step 3500: computing loss...
step 4000: computing loss...
step 4500: computing loss...
Final Loss: 2.4247
--- GENERATED TEXT ---

Wawice my.


DEROYom IINup
Yowhs, tof isth ble milendill, bes ireeesen cin latistt drov te, ando m p.


Wilerans!
el lind me llllishe cechiry: tupr aisspllw y.
Hen n'
I y fopetelaves homery wod mothak


In [16]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        # Create a list of 'Head' objects
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # A linear layer to merge the outputs back together
        self.proj = nn.Linear(num_heads * head_size, n_embd)

    def forward(self, x):
        # Run each head and concatenate the results along the last dimension
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        return out

In [17]:
n_embd = 64 # Increasing size for better "brain" capacity
n_head = 4  # 4 heads running in parallel (64 / 4 = 16 dimensions per head)

class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
        )
    def forward(self, x):
        return self.net(x)

class TransformerModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # Multi-head attention!
        self.sa_heads = MultiHeadAttention(n_head, n_embd // n_head)
        self.ffwd = FeedForward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # Combine token meaning + position meaning
        tok_emb = self.token_embedding_table(idx) # (B, T, C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, C)
        x = tok_emb + pos_emb # (B, T, C)

        # Look (Attention) -> Think (FeedForward)
        x = self.sa_heads(x)
        x = self.ffwd(x)

        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [18]:
import torch.nn as nn

class Block(nn.Module):
    """ Transformer block: communication followed by computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        # x + ... is the Residual Connection
        x = x + self.sa(self.ln1(x)) # Norm before Attention
        x = x + self.ffwd(self.ln2(x)) # Norm before FeedForward
        return x

In [19]:
# 1. Initialize the upgraded model
# Make sure your 'Head' class and new 'BigramLanguageModel' class are defined above this!
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = BigramLanguageModel(vocab_size)
m = model.to(device)

# 2. Set up the optimizer for the new parameters
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

# 3. The Training Loop
max_iters = 5000
eval_interval = 500

for iter in range(max_iters):

    # Every now and then evaluate the loss on train and val sets
    if iter % eval_interval == 0:
        print(f"step {iter}: computing loss...")

    # Sample a batch of data
    xb, yb = get_batch('train')

    # Move data to the correct device
    xb = xb.to(device)
    yb = yb.to(device)

    # Evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"Final Loss: {loss.item():.4f}")

# 4. Generate something!
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("--- GENERATED TEXT ---")
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))

step 0: computing loss...
step 500: computing loss...
step 1000: computing loss...
step 1500: computing loss...
step 2000: computing loss...
step 2500: computing loss...
step 3000: computing loss...
step 3500: computing loss...
step 4000: computing loss...
step 4500: computing loss...
Final Loss: 2.5578
--- GENERATED TEXT ---


ANRWindo ourtCeiiby we atit,
CHive wenghiee s psousower; te
To kidanthrupirf son; igist m:
Et Ce Raleronth, af Prr?

WISo myr fu, be!
s,
Sby ak
Sadsal this ghe tohoin couk ayraney Iry tsthofr t ce.
J


In [20]:
# --- Updated Hyperparameters for a "Heavy" Training Run ---
batch_size = 64     # How many sequences to process in parallel
block_size = 256    # Maximum context length (bigger = more "memory")
max_iters = 5000    # Total training steps
learning_rate = 3e-4
n_embd = 384        # Every character is represented by a 384-length vector
n_head = 6          # 384 / 6 = 64 dimensions per head
n_layer = 6         # We stack 6 Blocks on top of each other
dropout = 0.2       # 20% of neurons turn off randomly to prevent memorization
# ---------------------------------------------------------

class TransformerModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)

        # This creates the "Deep" part of the network
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])

        # Final layer normalization before the output
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        tok_emb = self.token_embedding_table(idx)
        # Using arange up to T ensures we match the current sequence length
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb

        x = self.blocks(x) # Pass through the 6 stacked blocks
        x = self.ln_f(x)   # Final normalization
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [23]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import requests

# --- 1. Hyperparameters ---
batch_size = 64      # How many sequences to process in parallel
block_size = 256     # Context length (how many characters the model looks at)
max_iters = 5000     # Training steps
eval_interval = 500  # How often to check loss
learning_rate = 3e-4 # How fast we tweak the weights
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 384         # Vector size for each character
n_head = 6           # Number of attention heads
n_layer = 6          # Number of Transformer Blocks
dropout = 0.2        # Probability of dropping neurons
# ---------------------------

torch.manual_seed(1337)

# --- 2. Data Loading & Tokenization ---
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
text = requests.get(url).text

chars = sorted(list(set(text)))
vocab_size = len(chars)
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[i] for i in l])

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9*len(data))
train_data = data[:n]
val_data = data[n:]

def get_batch(split):
    data_set = train_data if split == 'train' else val_data
    ix = torch.randint(len(data_set) - block_size, (batch_size,))
    x = torch.stack([data_set[i:i+block_size] for i in ix])
    y = torch.stack([data_set[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# --- 3. Model Architecture ---

class Head(nn.Module):
    """ One head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B,T,C = x.shape
        k = self.key(x)
        q = self.query(x)
        wei = q @ k.transpose(-2,-1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        v = self.value(x)
        out = wei @ v
        return out

class MultiHeadAttention(nn.Module):
    """ Multiple heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    """ A simple linear layer followed by non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head=n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# --- 4. Initialization & Training ---
model = BigramLanguageModel()
m = model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

print(f"Total Parameters: {sum(p.numel() for p in m.parameters())/1e6:.2f}M")

for iter in range(max_iters):
    if iter % eval_interval == 0 or iter == max_iters - 1:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

# --- 5. Final Generation ---
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print("\n--- GENERATED SHAKESPEARE ---\n")
print(decode(m.generate(context, max_new_tokens=500)[0].tolist()))

Total Parameters: 10.79M
step 0: train loss 4.2849, val loss 4.2823
step 500: train loss 2.0112, val loss 2.0971
step 1000: train loss 1.6021, val loss 1.7830
step 1500: train loss 1.4412, val loss 1.6396
step 2000: train loss 1.3430, val loss 1.5724
step 2500: train loss 1.2809, val loss 1.5330
step 3000: train loss 1.2268, val loss 1.5094
step 3500: train loss 1.1824, val loss 1.4881
step 4000: train loss 1.1475, val loss 1.4869
step 4500: train loss 1.1108, val loss 1.4805
step 4999: train loss 1.0779, val loss 1.4920

--- GENERATED SHAKESPEARE ---


But with prison: I will stead with you.

ISABELLA:
Carress, all do; and I'll say your honour self good:
Then I'll regn your highness and
Compell'd by my sweet gates that you may:
Valiant make how I heard of you.

ANGELO:
Nay, sir, Isay!

ISABELLA:
I am sweet men sister as you steed.

LUCIO:
As it if you in the case would princily,
I'll rote, sir, I did cannot now at me?
That look thence, thy children shall be you called.

DUKE VINCENTIO

In [24]:
# Save the model's learned parameters to a file
torch.save(model.state_dict(), 'mini_gpt_shakespeare.pth')
print("Model saved to 'mini_gpt_shakespeare.pth'")

Model saved to 'mini_gpt_shakespeare.pth'


In [25]:
from google.colab import files
files.download('mini_gpt_shakespeare.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [26]:
# 1. Create a fresh instance of the model architecture
loaded_model = BigramLanguageModel()

# 2. Load the weights from the file (make sure the file is uploaded to Colab)
loaded_model.load_state_dict(torch.load('mini_gpt_shakespeare.pth'))

# 3. Move it to the GPU and set to evaluation mode
loaded_model.to(device)
loaded_model.eval()

print("Model successfully reloaded and ready to generate!")

Model successfully reloaded and ready to generate!


In [27]:
def chat_with_model(model, max_new_tokens=100):
    model.eval() # Switch to evaluation mode
    print("--- Start Chatting! (type 'quit' to stop) ---")

    while True:
        user_input = input("You: ")
        if user_input.lower() == 'quit':
            break

        # Format the input like a script
        prompt = f"\nUSER: {user_input}\nAI:"

        # Encode the prompt
        context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)

        # Generate the response
        generated_tokens = model.generate(context, max_new_tokens=max_new_tokens)[0].tolist()
        full_response = decode(generated_tokens)

        # Extract only the new part (the AI's answer)
        # We split by 'AI:' and take the last part to keep the UI clean
        ai_response = full_response[len(prompt):].split('\n')[0]

        print(f"Mini-GPT: {ai_response}")

# Call the function
chat_with_model(m)

--- Start Chatting! (type 'quit' to stop) ---
You: Your name?
Mini-GPT:  la, good tree morrow, get thee to answer me.
You: Dinner?
Mini-GPT: 
You: William
Mini-GPT:  why, provost! what art thou these swift on,
You: Queen
Mini-GPT: 
You: Romeo
Mini-GPT: 
You: Who are you?
Mini-GPT: 
You: What is for dinner?
Mini-GPT: 


KeyboardInterrupt: Interrupted by user

In [28]:
!pip install gradio

In [29]:
import gradio as gr

def respond(message, history):
    prompt = f"USER: {message}\nAI:"
    context = torch.tensor(encode(prompt), dtype=torch.long, device=device).unsqueeze(0)
    output = decode(model.generate(context, max_new_tokens=50)[0].tolist())
    return output[len(prompt):].split('\n')[0]

gr.ChatInterface(respond).launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2866dade5de9b76f36.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
